In [1]:
import pandas as pd
import numpy as np
from datetime import datetime
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

import warnings
warnings.filterwarnings('ignore')

# AI를 활용하여 상황가정해서 A/B Test 시뮬레이션 간단하게, 빠르게 돌려보기

### 1) 퍼널 분석
- 당신은 대출중개플랫폼의 신용대출 제품 DA입니다.
- 최근 대출 실행률(대출 실행 / 대출신청)이 감소하는 것을 대시보드를 통해서 확인했습니다.
- 문제를 찾던 도중, 특정 제휴사의 대출실행률이 유난히 떨어지는 것을 발견했습니다.
- 해당 제휴사의 대출 실행 퍼널 중 **어떤 퍼널에서 문제가 발생한 것**인지 분석해보세요.

 A: 본인인증 >> B: 신분증인증 >> C: 개인정보입력 >> D: 대출 조건 기입(상환방법, 상환기간, 대출금액 등) >> E: 대출 심사 >> F: 대출 실행

In [3]:
funnel_df = pd.read_csv('../data/abtest1_data.csv')

In [4]:
funnel_df

,user_id,screen_name,log_time
0,user_1,A,2024-01-01 00:00:00
1,user_1,B,2024-01-01 00:01:00
2,user_1,C,2024-01-01 00:02:00
3,user_1,D,2024-01-01 00:04:00
4,user_1,E,2024-01-01 00:07:00
...,...,...,...
25694,user_9998,B,2024-01-01 00:01:00
25695,user_9999,A,2024-01-01 00:00:00
25696,user_9999,B,2024-01-01 00:03:00
25697,user_10000,A,2024-01-01 00:00:00


In [7]:
agg_funnel = funnel_df.groupby('screen_name')['user_id'].nunique().reset_index()
agg_funnel


,screen_name,user_id
0,A,10000
1,B,8024
2,C,2482
3,D,2213
4,E,1762
5,F,1218


In [8]:
agg_funnel.rename(columns={'user_id':'user_cnt'}, inplace=True)
agg_funnel['conversion'] = agg_funnel['user_cnt'] / agg_funnel.loc[0, 'user_cnt']
agg_funnel

,screen_name,user_cnt,conversion
0,A,10000,1.0000
1,B,8024,0.8024
2,C,2482,0.2482
3,D,2213,0.2213
4,E,1762,0.1762
5,F,1218,0.1218


### B -> C 에서 심각한 이탈이 일어남.. 유관부서에 리포팅하고 UX를 개선하여 A/B test를 해봐야하지 않을까?

### 2) 문제 정의 & 실험
- 당신은 신분증 인증 화면에서 대다수의 유저가 이탈하는 것을 팀의 PO(Product Owner)와 FE(프론트 개발자)에게 리포팅 했습니다.
- 확인 결과, 제휴사가 사용하는 신분증 촬영 & 인식 솔루션에 심각한 결함이 있는 것을 확인했고, 제휴사는 결함 해결을 위한 개발 작업에 착수했습니다.
- PD(Product Designer)는 이 참에 UX 개선도 함께 할 것을 제안하였고 팀원 모두가 이에 대해 동의했습니다.
- 당신은 실험군과 대조군을 나눠 각각의 그룹에게 다른 화면을 보여주는 실험을 설계했고, 제휴사 측에서 실험까지 반영해서 웹을 재배포 했습니다.
- **실험 결과**를 분석해주세요.

- 실험군: 화면 A 노출 유저
- 대조군: 기존 화면 노출 유저
- 개선지표: C 화면 도달률

In [10]:
abtest_df = pd.read_csv('../data/abtest2_data.csv')
abtest_df

,user_id,screen_name,log_time,group
0,user_1,A,2024-01-01 00:00:00,A
1,user_1,B,2024-01-01 00:01:00,A
2,user_1,C,2024-01-01 00:02:00,A
3,user_1,D,2024-01-01 00:04:00,A
4,user_1,E,2024-01-01 00:07:00,A
...,...,...,...,...
56346,user_9998,A,2024-01-01 00:00:00,B
56347,user_9998,B,2024-01-01 00:03:00,B
56348,user_9999,A,2024-01-01 00:00:00,B
56349,user_9999,B,2024-01-01 00:03:00,B


In [12]:
abtest_df.groupby(['group', 'screen_name'])['user_id'].nunique()


group  screen_name
A      A              10000
       B               7610
       C               3952
       D               3555
       E               2836
       F               2017
B      A              10000
       B               7914
       C               2735
       D               2424
       E               1945
       F               1363
Name: user_id, dtype: int64

In [13]:
abtest_df[abtest_df['screen_name'] == 'B'][['user_id', 'group']]

,user_id,group
1,user_1,A
7,user_2,A
13,user_3,A
20,user_5,A
22,user_6,A
...,...,...
56337,user_9994,B
56342,user_9995,B
56345,user_9997,B
56347,user_9998,B


In [14]:
scr_b = abtest_df[abtest_df['screen_name'] == 'B'][['user_id', 'group']]
scr_c = abtest_df[abtest_df['screen_name'] == 'C'][['user_id', 'group']]
scr_c['exp_c_yn'] = 1

In [17]:
scr_b

,user_id,group
1,user_1,A
7,user_2,A
13,user_3,A
20,user_5,A
22,user_6,A
...,...,...
56337,user_9994,B
56342,user_9995,B
56345,user_9997,B
56347,user_9998,B


In [18]:
scr_c

,user_id,group,exp_c_yn
2,user_1,A,1
8,user_2,A,1
14,user_3,A,1
23,user_6,A,1
27,user_8,A,1
...,...,...,...
56311,user_9986,B,1
56317,user_9989,B,1
56323,user_9990,B,1
56328,user_9991,B,1


In [21]:
abtest = pd.merge(scr_b, scr_c, how='left', on=['user_id', 'group'])
abtest.fillna(0, inplace=True)
abtest

,user_id,group,exp_c_yn
0,user_1,A,1.0
1,user_2,A,1.0
2,user_3,A,1.0
3,user_5,A,0.0
4,user_6,A,1.0
...,...,...,...
15519,user_9994,B,1.0
15520,user_9995,B,0.0
15521,user_9997,B,0.0
15522,user_9998,B,0.0


In [23]:
conv_a = abtest[abtest['group']=='A'].exp_c_yn.sum() / abtest[abtest['group']=='A'].user_id.nunique()
conv_b = abtest[abtest['group']=='B'].exp_c_yn.sum() / abtest[abtest['group']=='B'].user_id.nunique()

In [24]:
print(f"A안 전환율: {round(conv_a*100, 2)}%")
print(f"B안 전환율: {round(conv_b*100, 2)}%")
print(f"전환율 개선율: {round((conv_a - conv_b) / conv_b * 100, 2)}%")

A안 전환율: 51.93%
B안 전환율: 34.56%
전환율 개선율: 50.27%


### **'C까지 도달했는지의 여부'** 를 따지는 범주형 변수이므로 카이제곱 검정 채택

In [22]:
contingency_table = pd.crosstab(abtest['group'], abtest['exp_c_yn'])

# 카이제곱 검정
chi2_stat, p_value_chi2, dof, expected = stats.chi2_contingency(contingency_table)

# 결과 출력
print(f"Chi-squared statistic: {chi2_stat}")
print(f"P-value: {p_value_chi2}")
print(f"Degrees of freedom: {dof}")

Chi-squared statistic: 476.7999012015489
P-value: 1.0618980557855836e-105
Degrees of freedom: 1


- A안(UX 개선)의 전환율이 51.93%로 B안(기존)의 34.56%보다 약 17.4%p 높고, 이는 기존 대비 50.27%의 개선. 

- p-value가 사실상 0이므로 이 차이가 우연일 가능성은 없고, UX 개선이 'C 도달률'을 확실히 높였다고 결론 내릴 수 있음.